In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(".")

POS_DIR = PROJECT_ROOT / "Data_proc" / "positives"
NEG_DIR = PROJECT_ROOT / "Data_proc" / "negatives"
INTERIM_NEG_DIR = PROJECT_ROOT / "Data_interim" / "negatives"
QC_DIR = PROJECT_ROOT / "Data_proc" / "qc_reports"

for d in [NEG_DIR, INTERIM_NEG_DIR, QC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

E3_POS_PATH = POS_DIR / "positive_e3.csv"
DUB_POS_PATH = POS_DIR / "positive_dub.csv"
ALL_POS_PATH = POS_DIR / "positive_all.csv"

NEG_PER_POS = 2.0
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

e3_pos = pd.read_csv(E3_POS_PATH)
dub_pos = pd.read_csv(DUB_POS_PATH)
all_pos = pd.read_csv(ALL_POS_PATH)

print("E3 positive:", e3_pos.shape)
print("DUB positive:", dub_pos.shape)
print("ALL positive:", all_pos.shape)

In [ ]:
def validate_positive_df(df: pd.DataFrame, enzyme_class: str):
    required_cols = [
        "pair_id",
        "group_id",
        "enzyme_class",
        "enz_ac",
        "sub_ac",
        "enz_gene",
        "sub_gene",
        "enzyme_type",
        "label",
        "source",
        "pmid",
    ]
    
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"{enzyme_class}: missing columns: {missing_cols}")
    
    if df["enzyme_class"].isna().any():
        raise ValueError(f"{enzyme_class}: enzyme_class has NaN values.")
    
    bad_class = set(df["enzyme_class"].dropna().unique()) - {enzyme_class}
    if bad_class:
        raise ValueError(f"{enzyme_class}: unexpected enzyme_class values: {bad_class}")
    
    label_values = set(df["label"].dropna().astype(int).unique())
    if label_values != {1}:
        raise ValueError(f"{enzyme_class}: positive df must have label=1 only. Found: {label_values}")
    
    if df["pair_id"].duplicated().any():
        raise ValueError(f"{enzyme_class}: duplicated pair_id detected in positives.")
    
    if df["pair_id"].astype(str).str.startswith("nan|").any():
        raise ValueError(f"{enzyme_class}: invalid pair_id starts with nan|.")
    
    print(f"{enzyme_class}: positive validation passed.")


def rebuild_ids(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    
    out["enz_ac"] = out["enz_ac"].astype(str).str.strip()
    out["sub_ac"] = out["sub_ac"].astype(str).str.strip()
    out["enzyme_class"] = out["enzyme_class"].astype(str).str.strip()
    
    out["pair_id"] = (
        out["enzyme_class"]
        + "|"
        + out["enz_ac"]
        + "|"
        + out["sub_ac"]
    )
    
    out["group_id"] = (
        out["enzyme_class"]
        + "|"
        + out["enz_ac"]
    )
    
    return out


def make_candidate_pairs(df_pos: pd.DataFrame, enzyme_class: str) -> pd.DataFrame:
    """
    Build all enzyme × substrate candidate pairs for one enzyme class,
    then remove known positive pairs.
    """
    pos = df_pos.copy()
    
    enzymes = (
        pos[["enz_ac", "enz_gene", "enzyme_type"]]
        .drop_duplicates(subset=["enz_ac"])
        .reset_index(drop=True)
    )
    
    substrates = (
        pos[["sub_ac", "sub_gene"]]
        .drop_duplicates(subset=["sub_ac"])
        .reset_index(drop=True)
    )
    
    enzymes["_key"] = 1
    substrates["_key"] = 1
    
    candidates = (
        enzymes.merge(substrates, on="_key", how="inner")
        .drop(columns="_key")
    )
    
    candidates["enzyme_class"] = enzyme_class
    candidates["label"] = 0
    candidates["source"] = "initial_negative_candidate"
    candidates["pmid"] = "NA"
    
    candidates = rebuild_ids(candidates)
    
    positive_pair_ids = set(pos["pair_id"].astype(str))
    candidates = candidates[
        ~candidates["pair_id"].astype(str).isin(positive_pair_ids)
    ].copy()
    
    cols = [
        "pair_id",
        "group_id",
        "enzyme_class",
        "enz_ac",
        "sub_ac",
        "enz_gene",
        "sub_gene",
        "enzyme_type",
        "label",
        "source",
        "pmid",
    ]
    
    candidates = (
        candidates[cols]
        .drop_duplicates(subset=["pair_id"])
        .reset_index(drop=True)
    )
    
    return candidates


def sample_negatives_degree_matched(
    df_pos: pd.DataFrame,
    candidates: pd.DataFrame,
    n_target: int,
    neg_per_pos: float = 2.0,
    random_seed: int = 42,
) -> pd.DataFrame:
    """
    Degree-aware negative sampling.
    For each positive row, sample negatives for the same enzyme when possible.
    This approximately preserves enzyme frequency.
    """
    rng = np.random.default_rng(random_seed)
    
    pos = df_pos.copy()
    candidates = candidates.copy()
    
    cand_by_enzyme = {
        enz: sub_df.copy()
        for enz, sub_df in candidates.groupby("enz_ac")
    }
    
    sampled_rows = []
    used_pair_ids = set()
    
    n_per_positive_floor = int(np.floor(neg_per_pos))
    fractional_part = neg_per_pos - n_per_positive_floor
    
    for _, row in pos.iterrows():
        enz_ac = row["enz_ac"]
        
        k = n_per_positive_floor
        if fractional_part > 0 and rng.random() < fractional_part:
            k += 1
        
        enzyme_candidates = cand_by_enzyme.get(enz_ac)
        
        if enzyme_candidates is None or len(enzyme_candidates) == 0:
            continue
        
        available = enzyme_candidates[
            ~enzyme_candidates["pair_id"].isin(used_pair_ids)
        ]
        
        if len(available) == 0:
            continue
        
        take = min(k, len(available))
        picked_idx = rng.choice(available.index.to_numpy(), size=take, replace=False)
        picked = available.loc[picked_idx]
        
        for _, neg_row in picked.iterrows():
            sampled_rows.append(neg_row)
            used_pair_ids.add(neg_row["pair_id"])
    
    sampled = pd.DataFrame(sampled_rows)
    
    if len(sampled) < n_target:
        remaining_needed = n_target - len(sampled)
        remaining_pool = candidates[
            ~candidates["pair_id"].isin(used_pair_ids)
        ]
        
        if len(remaining_pool) < remaining_needed:
            raise ValueError(
                f"Not enough remaining candidates. needed={remaining_needed}, "
                f"available={len(remaining_pool)}"
            )
        
        extra = remaining_pool.sample(
            n=remaining_needed,
            replace=False,
            random_state=random_seed + 999,
        )
        
        sampled = pd.concat([sampled, extra], ignore_index=True)
    
    if len(sampled) > n_target:
        sampled = sampled.sample(
            n=n_target,
            replace=False,
            random_state=random_seed + 123,
        )
    
    sampled = sampled.drop_duplicates(subset=["pair_id"]).reset_index(drop=True)
    sampled["source"] = "neg2x_initial_degree_matched"
    
    return sampled


def qc_negative_set(
    neg_df: pd.DataFrame,
    pos_df: pd.DataFrame,
    name: str,
) -> dict:
    neg_pair_ids = set(neg_df["pair_id"].astype(str))
    pos_pair_ids = set(pos_df["pair_id"].astype(str))
    
    conflicts = neg_pair_ids & pos_pair_ids
    
    return {
        "dataset": name,
        "n_rows": len(neg_df),
        "n_unique_pair_id": neg_df["pair_id"].nunique(),
        "n_duplicate_pair_id_rows": int(neg_df.duplicated("pair_id").sum()),
        "n_unique_group_id": neg_df["group_id"].nunique(),
        "n_unique_enz_ac": neg_df["enz_ac"].nunique(),
        "n_unique_sub_ac": neg_df["sub_ac"].nunique(),
        "n_missing_enzyme_class": int(neg_df["enzyme_class"].isna().sum()),
        "n_missing_enz_ac": int(neg_df["enz_ac"].isna().sum()),
        "n_missing_sub_ac": int(neg_df["sub_ac"].isna().sum()),
        "n_pair_id_starts_with_nan": int(neg_df["pair_id"].astype(str).str.startswith("nan|").sum()),
        "n_label_0": int((neg_df["label"] == 0).sum()),
        "n_conflict_with_positive": len(conflicts),
    }


print("All functions are defined.")

سلول ۳: ساخت candidateها و neg2x

In [ ]:
validate_positive_df(e3_pos, "E3")
validate_positive_df(dub_pos, "DUB")

e3_candidates = make_candidate_pairs(e3_pos, "E3")
dub_candidates = make_candidate_pairs(dub_pos, "DUB")

print("E3 candidates:", e3_candidates.shape)
print("DUB candidates:", dub_candidates.shape)

e3_candidates.to_csv(
    INTERIM_NEG_DIR / "candidate_negatives_e3.csv",
    index=False
)

dub_candidates.to_csv(
    INTERIM_NEG_DIR / "candidate_negatives_dub.csv",
    index=False
)

n_e3_target = int(len(e3_pos) * NEG_PER_POS)
n_dub_target = int(len(dub_pos) * NEG_PER_POS)

print("E3 target negatives:", n_e3_target)
print("DUB target negatives:", n_dub_target)

e3_neg = sample_negatives_degree_matched(
    df_pos=e3_pos,
    candidates=e3_candidates,
    n_target=n_e3_target,
    neg_per_pos=NEG_PER_POS,
    random_seed=RANDOM_SEED,
)

dub_neg = sample_negatives_degree_matched(
    df_pos=dub_pos,
    candidates=dub_candidates,
    n_target=n_dub_target,
    neg_per_pos=NEG_PER_POS,
    random_seed=RANDOM_SEED + 1,
)

negative_all = pd.concat([e3_neg, dub_neg], ignore_index=True)

print("E3 neg:", e3_neg.shape)
print("DUB neg:", dub_neg.shape)
print("ALL neg:", negative_all.shape)

display(e3_neg.head())
display(dub_neg.head())

سلول ۴: QC و conflict check

In [ ]:
qc_rows = [
    qc_negative_set(e3_neg, e3_pos, "E3_negative_initial_2x"),
    qc_negative_set(dub_neg, dub_pos, "DUB_negative_initial_2x"),
    qc_negative_set(negative_all, all_pos, "ALL_negative_initial_2x"),
]

qc = pd.DataFrame(qc_rows)
display(qc)

# Conflict details
e3_conflict = e3_neg[e3_neg["pair_id"].isin(set(e3_pos["pair_id"]))]
dub_conflict = dub_neg[dub_neg["pair_id"].isin(set(dub_pos["pair_id"]))]
all_conflict = negative_all[negative_all["pair_id"].isin(set(all_pos["pair_id"]))]

print("E3 conflict rows:", len(e3_conflict))
print("DUB conflict rows:", len(dub_conflict))
print("ALL conflict rows:", len(all_conflict))

display(e3_conflict.head())
display(dub_conflict.head())
display(all_conflict.head())

# Additional label sanity
print("E3 label counts:")
print(e3_neg["label"].value_counts(dropna=False))

print("\nDUB label counts:")
print(dub_neg["label"].value_counts(dropna=False))

print("\nALL label counts:")
print(negative_all["label"].value_counts(dropna=False))

سلول ۵: ذخیره خروجی‌ها

In [ ]:
e3_neg.to_csv(
    NEG_DIR / "negative_e3_initial_2x.csv",
    index=False
)

dub_neg.to_csv(
    NEG_DIR / "negative_dub_initial_2x.csv",
    index=False
)

negative_all.to_csv(
    NEG_DIR / "negative_all_initial_2x.csv",
    index=False
)

qc.to_csv(
    QC_DIR / "negative_initial_qc.csv",
    index=False
)

# Save conflicts even if empty
e3_conflict.to_csv(
    QC_DIR / "negative_e3_positive_conflicts.csv",
    index=False
)

dub_conflict.to_csv(
    QC_DIR / "negative_dub_positive_conflicts.csv",
    index=False
)

all_conflict.to_csv(
    QC_DIR / "negative_all_positive_conflicts.csv",
    index=False
)

print("Saved:")
print(NEG_DIR / "negative_e3_initial_2x.csv")
print(NEG_DIR / "negative_dub_initial_2x.csv")
print(NEG_DIR / "negative_all_initial_2x.csv")
print(QC_DIR / "negative_initial_qc.csv")

In [ ]:
check_files = [
    NEG_DIR / "negative_e3_initial_2x.csv",
    NEG_DIR / "negative_dub_initial_2x.csv",
    NEG_DIR / "negative_all_initial_2x.csv",
    QC_DIR / "negative_initial_qc.csv",
]

for f in check_files:
    print(f.name, "exists:", f.exists())
    if f.exists() and f.suffix == ".csv":
        tmp = pd.read_csv(f)
        print("shape:", tmp.shape)